# Results Analysis — Cop vs Thief via MCP Servers

**Methodology note**: the code cells below are real, correct, ready-to-run
code against this project's actual modules — but this notebook has **not
been executed** to produce fresh output (per an explicit instruction to
avoid any runtime changes during this documentation pass). The numbers
quoted in the markdown cells are figures that were directly observed and
printed during this project's actual development/verification sessions
(see `docs/PROMPT_ENGINEERING_LOG.md`, `docs/PRD_decision_engine.md`,
`docs/PRD_mcp_orchestration.md`, `docs/PRD_gmail_reporting.md`) — they are
not invented, and are not claimed to be this notebook's own output.

Where this session's data is incomplete or ambiguous, that is stated
explicitly below rather than papered over with a confident-sounding number.

## 1. Q-learning training scalability

`services/decision/q_learning_training.py::train_q_tables` was benchmarked
on a 5x5 grid (`max_moves=25`, `max_barriers=5`) at increasing episode
counts during Stage 4 development. All three timings below were directly
observed (printed to the terminal) in that session.

In [ ]:
import time

from cop_thief_mcp.services.decision.q_learning_training import train_q_tables
from cop_thief_mcp.services.game.grid import Grid

grid = Grid(rows=5, cols=5)

for episodes in (2000, 5000, 10000):
    start = time.time()
    cop_table, thief_table = train_q_tables(grid, max_moves=25, max_barriers=5, episodes=episodes)
    elapsed = time.time() - start
    print(f"episodes={episodes:6d}  elapsed={elapsed:6.2f}s  cop_states={len(cop_table.values):4d}  thief_states={len(thief_table.values):4d}")

### Observed results (from the actual development session)

| Episodes | Elapsed time | Notes |
|---|---|---|
| 2,000  | 2.51s | initial benchmark; cop table reached 86 (state,action) entries |
| 5,000  | 5.65s | run with default epsilon after adding start-position randomization |
| 10,000 | 9.65s | run with epsilon=0.25 (higher exploration) |

Roughly **~1ms of training time per episode** on this hardware, scaling
close to linearly with episode count on a 5x5 grid — fast enough to train
lazily once at MCP server startup (`servers/common.py::build_agent_server`
does exactly this when `decision_policy` is `"q_learning"`).

## 2. Real full-pipeline verifications (not a systematic sweep)

Two *real*, full end-to-end runs (real MCP servers, no mocking of the
decision policy) were performed and directly observed during this
project's Stage 5 and Stage 8 verification passes. Both happened to start
at the maximum possible separation on a 3x3 grid (Manhattan distance 4,
i.e. opposite corners of that small grid).

| Stage | Policy | Grid | Start (cop, thief) | Outcome | Moves | Cop score | Thief score |
|---|---|---|---|---|---|---|---|
| 5 (real OpenAI) | `llm` | 3x3 | (0,0), (2,2) | COP_WINS | 4 | 20 | 5 |
| 8 (real Gmail send) | `heuristic` | 3x3 | (0,0), (2,2) | COP_WINS | not printed by that run's script | 20 | 5 |

Both are genuine captures, confirming the full pipeline (engine + MCP
transport + decision policy, real OpenAI in one case) works correctly
end-to-end. These are **two data points, not a sensitivity sweep** — see
the honesty note below before generalizing from them.

## 3. Documented limitation: adjacent-start oscillation

Traced step-by-step during Stage 4 development (see
`docs/PRD_decision_engine.md`): starting the Cop and Thief at Manhattan
distance 1 (e.g. `cop=(2,2)`, `thief=(2,3)`) with the heuristic policy
settles into a **stable 2-round cycle** — the pair oscillates between two
fixed position pairs at distance 1 forever, never reaching distance 0,
even with 1-ply lookahead. This was directly observed by printing the
position pair every round for the full 25-move cap; every single round
showed distance 1, never 0. This is a known characteristic of
finite-lookahead pursuit strategies against an equally-reactive evader,
not an implementation bug — see the PRD for the full trace and reasoning.

## 4. Honesty note: an unresolved inconsistency, flagged rather than hidden

While preparing this notebook, an earlier batch of direct-engine tests
(run during Stage 4, testing several non-adjacent starting pairs
including opposite 5x5-grid corners) was recalled as showing
`THIEF_WINS` for *all* of them after the corner-pinch fix — which would
conflict with `docs/PRD_decision_engine.md`'s claim that "every other
tested starting separation (distance >= 2, including opposite grid
corners) captures reliably." The two *real, full-pipeline* verifications
above (section 2) did each capture successfully at distance 4 on a
**3x3** grid, which is some evidence for the heuristic working at that
distance on a small board — but that is not the same claim as reliable
capture from **5x5-grid opposite corners** (distance 8), which was not
re-confirmed with full confidence while writing this notebook.

**This has deliberately not been re-verified by running new code**, per
the instruction this notebook was written under. Recorded here as a
concrete, actionable follow-up rather than silently repeating a claim
that could not be re-confirmed with full confidence, or silently
"correcting" an already-committed PRD based on uncertain memory.

## 5. Follow-up work: a real systematic sensitivity sweep

The task doc's progressive sanity-check methodology (2x2 -> 3x3/3x2 ->
4x4/4x3 -> 5x5) suggests the right shape for resolving section 4's open
question: run many repeated trials at each Manhattan distance, at each
grid size, and tabulate capture rate. The cell below is ready-to-run
against the real `heuristic` policy (pure Python, no MCP/network needed)
to produce that table — intentionally left unexecuted here.

In [ ]:
from cop_thief_mcp.services.decision.heuristic import choose_cop_action, choose_thief_action
from cop_thief_mcp.services.game.grid import Grid, Position
from cop_thief_mcp.services.game.sub_game import TurnContext, run_sub_game


def cop_policy(ctx: TurnContext):
    return choose_cop_action(ctx.grid, ctx.own_position, ctx.opponent_position, ctx.barriers, ctx.barriers_placed, ctx.max_barriers)


def thief_policy(ctx: TurnContext):
    return choose_thief_action(ctx.grid, ctx.own_position, ctx.opponent_position, ctx.barriers)


def capture_rate_by_distance(grid: Grid, max_moves: int, trials_per_pair: int = 1) -> dict:
    """Sweep every (cop_start, thief_start) pair on `grid`, grouped by Manhattan
    distance, and report the fraction that end in a Cop capture.
    """
    from collections import defaultdict

    results = defaultdict(list)
    for cop_row in range(grid.rows):
        for cop_col in range(grid.cols):
            for thief_row in range(grid.rows):
                for thief_col in range(grid.cols):
                    cop_start = Position(cop_row, cop_col)
                    thief_start = Position(thief_row, thief_col)
                    if cop_start == thief_start:
                        continue
                    distance = abs(cop_row - thief_row) + abs(cop_col - thief_col)
                    result = run_sub_game(grid, cop_start, thief_start, cop_policy, thief_policy, max_moves, max_barriers=5)
                    results[distance].append(result.outcome.value == "cop_wins")

    return {d: sum(v) / len(v) for d, v in sorted(results.items())}


# Ready to run, e.g.:
# print(capture_rate_by_distance(Grid(rows=5, cols=5), max_moves=25))